<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/Analytics-Forecast/blob/main/Branch_TimeSeries_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("/content/ODSummary-CSV-AS-ON-04-06-2026.csv")

In [2]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df = df.set_index("Date")

columns_to_fill = [
    "CountofAccountID",
    "CountofCreditOfficerID",
    "SumofPrincipalOutstanding",
    "SumofInterestOutstanding",
    "SumofTotalPrincipalOverdue",
    "SumofTotalInterestOverdue",
    "SumofTotalPAR",
]

df_filled = (
    df.groupby("BranchID")[columns_to_fill]
    .resample("D")
    .ffill()
    .reset_index()
)

df_filled = df_filled[
    ["BranchID", "Date"]
    + [col for col in df_filled.columns if col not in ["BranchID", "Date"]]
]

In [3]:
df_filled['Day'] = df_filled['Date'].dt.day
df_filled['Month'] = df_filled['Date'].dt.month
df_filled['Year'] = df_filled['Date'].dt.year
df_filled['DayOfWeek'] = df_filled['Date'].dt.dayofweek

In [ ]:
features_to_lag = [
    'CountofAccountID',
    'CountofCreditOfficerID', 'SumofPrincipalOutstanding',
    'SumofInterestOutstanding', 'SumofTotalPrincipalOverdue',
    'SumofTotalInterestOverdue', 'SumofTotalPAR'
]

lags = [1, 7, 15, 30, 365]

for lag in lags:
    for col in features_to_lag:
        lag_col_name = f"{col}_{lag}day_lag"

        df_filled[lag_col_name] = df_filled.groupby('BranchID')[col].shift(lag)

        df_filled[lag_col_name] = df_filled[lag_col_name].fillna(df_filled[col])

print(df_filled.head())

df_filled.to_csv('ml_ready_branch_data.csv')

   BranchID       Date  CountofAccountID  CountofCreditOfficerID  \
0         1 2024-02-01               314                     314   
1         1 2024-02-02               314                     314   
2         1 2024-02-03               314                     314   
3         1 2024-02-04               314                     314   
4         1 2024-02-05               314                     314   

   SumofPrincipalOutstanding  SumofInterestOutstanding  \
0                    4078796                    405923   
1                    4078796                    405923   
2                    4078796                    405923   
3                    4078796                    405923   
4                    4078796                    405923   

   SumofTotalPrincipalOverdue  SumofTotalInterestOverdue  SumofTotalPAR  \
0                     3496538                     366587        4078796   
1                     3503319                     367046        4078796   
2                